In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':20})
import os
from scipy.io import loadmat

import sys
sys.path.append('/home/ks2823/SerialDilutionModel/src/')

from Ophelia_evaluation import LatentAssemblyModel, load_venturelli_arrays, train_model, evaluate_predictions
import torch
import copy
from glob import glob

In [8]:
def extract_rumen_trajectories(metadata_df, species_df, save_path=None):
    """
    Parses the Rumen enrichment metadata and species counts to extract chronological 
    sequences of relative abundances for every unique experimental condition.
    
    Returns:
        dict: Keys are unique condition tuples (Origin, Substrate, Pressure).
              Values are chronologically ordered PyTorch tensors of relative abundances 
              (Shape: Timepoints x Species).
        list: Strains/Taxonomy names in the correct index order.
    """
    # 0. Process Species Data (Normalize to relative abundance)
    # Find the taxonomy column (case-insensitive search)
    tax_col = [c for c in species_df.columns if 'tax' in c.lower()][0]
    
    # Set taxonomy as index and ensure all data is numeric
    species_counts = species_df.set_index(tax_col).apply(pd.to_numeric, errors='coerce').fillna(0)
    
    # Normalize columns (samples) so they sum to 1
    rel_abund = species_counts.div(species_counts.sum(axis=0), axis=1)
    rel_abund = rel_abund.fillna(0.0) # Handle completely empty samples if any
    strains = rel_abund.index.tolist()
    
    # 1. Clean up DataFrame and skip missing Transfers (Inoculum/T0)
    df = metadata_df.copy()
    
    substrate_col = 'Methanogenic  substrate'
    pressure_col = 'Hydrogen partial pressure (psi)'
    origin_col = 'Rumen sample origin'
    
    # Drop any rows where 'Transfer' is NaN (the Inoculum) or explicitly 'T0'
    df = df.dropna(subset=['Transfer'])
    df = df[df['Transfer'] != 'T0']
    
    # 2. Group by Unique Conditions (Trajectories now start at T1)
    trajectories = {}
    grouped = df.groupby([origin_col, substrate_col, pressure_col], dropna=False)
    
    for condition_tuple, group_data in grouped:
        origin, substrate, pressure = condition_tuple
        
        # Sort the group by Transfer (T1, T2, T3, etc.)
        group_data = group_data.copy()
        group_data['T_num'] = group_data['Transfer'].str.extract(r'(\d+)').astype(int)
        sorted_group = group_data.sort_values('T_num')
        
        # Extract the sequence of treated Sample IDs (T1 onwards)
        sequence = sorted_group['Sample ID'].tolist()
        
        # 3. Map Sample IDs to Relative Abundance vectors
        valid_sequence = []
        for sample_id in sequence:
            if sample_id in rel_abund.columns:
                valid_sequence.append(rel_abund[sample_id].values)
            else:
                print(f"Warning: Sample ID '{sample_id}' not found in species data. Skipping.")
        
        # Save if we have at least 2 points (e.g. T1 -> T2)
        if len(valid_sequence) > 1:
            # Convert to PyTorch Tensor of shape (Time_steps, Num_species)
            trajectories[condition_tuple] = torch.tensor(np.array(valid_sequence), dtype=torch.float32)
        
    # 4. Save outputs
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save({'trajectories': trajectories, 'strains': strains}, save_path)
        print(f"Saved extracted relative abundance trajectories to '{save_path}'")
        
    return trajectories, strains

In [3]:
data_dir = '../Data/'

species_data_path = os.path.join(data_dir, 'species.txt')

In [4]:
species_data = pd.read_csv(species_data_path, sep='\t')
species_data = species_data.rename(columns={'Unnamed: 0': 'taxonomy'})
species_data.head()

,taxonomy,BE-1010-T1-M05,BE-1010-T1-M30,BE-1010-T1-N05,BE-1010-T1-N30,BE-1010-T1-T05,BE-1010-T1-T30,BE-1010-T2-M05,BE-1010-T2-M30,BE-1010-T2-N05,...,BE-1011-T5-N05,BE-1011-T5-N30,BE-1011-T5-T05,BE-1011-T5-T30,BE-1011-T6-M05,BE-1011-T6-M30,BE-1011-T6-N05,BE-1011-T6-N30,BE-1011-T6-T05,BE-1011-T6-T30
0,Archaea|Euryarchaeota|Methanomicrobia|Methanom...,0.414132,0.381383,0.281651,0.210417,0.007401,0.037288,0.041557,0.177807,0.006092,...,0.065360,0.423262,0.004218,0.000031,0.000000,0.009909,0.418829,0.546549,0.215175,0.000000
1,Bacteria|Synergistetes|Synergistia|Synergistal...,0.090022,0.101543,0.051995,0.095122,0.075380,0.071628,0.075582,0.044339,0.045201,...,0.020132,0.013392,0.034567,0.019355,0.008291,0.004620,0.013668,0.005357,0.018427,0.015606
2,Bacteria|Firmicutes|Bacilli|Lactobacillales|St...,0.072366,0.039241,0.076887,0.000067,0.108731,0.090901,0.052932,0.043863,0.113838,...,0.022593,0.025737,0.070785,0.011048,0.002917,0.004432,0.006991,0.008627,0.025881,0.094910
3,Bacteria|Bacteroidetes|Bacteroidia|Bacteroidal...,0.044427,0.014155,0.017633,0.009081,0.012106,0.006673,0.037508,0.016198,0.021384,...,0.040906,0.014481,0.009235,0.028010,0.020659,0.008922,0.065792,0.009500,0.013495,0.015160
4,Bacteria|Synergistetes|Synergistia|Synergistal...,0.025220,0.027153,0.015551,0.029526,0.022796,0.022021,0.023352,0.013682,0.013627,...,0.014326,0.005679,0.023916,0.015235,0.003579,0.002414,0.008760,0.003341,0.014624,0.008886


In [5]:
rumen_enrichment_metadata_path = os.path.join(data_dir, 'Rumen enrichment metadata.xlsx')
rumen_enrichment_metadata = pd.read_excel(rumen_enrichment_metadata_path)
rumen_enrichment_metadata = rumen_enrichment_metadata.drop(columns=['Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7'])
rumen_enrichment_metadata.head()

,Sample ID,Rumen sample origin,Transfer,Methanogenic substrate,Hydrogen partial pressure (psi)
0,BE-1010-T1-T05,OSU,T1,H2:CO2 + TMA,5.0
1,BE-1011-T1-N30,BRRS,T1,H2:CO2,30.0
2,BE-1010-T1-N05,OSU,T1,H2:CO2,5.0
3,BE-1011-T1-M30,BRRS,T1,H2:CO2 + MeOH,30.0
4,BE-1011-T1-T30,BRRS,T1,H2:CO2 + TMA,30.0


In [11]:
condition_trajectories, strain_list = extract_rumen_trajectories(rumen_enrichment_metadata, species_data, save_path='../Data/German_processed_data.pt')

Saved extracted relative abundance trajectories to '../Data/German_processed_data.pt'


In [19]:
for cond, seq_tensor in condition_trajectories.items():
    print(f"Condition: {cond}")
    # print(f"Shape (Timepoints, Species): {seq_tensor.shape}")
    # print(f"Sequence Tensor:\n{seq_tensor}")

Condition: ('BRRS', 'H2:CO2', np.float64(5.0))
Condition: ('BRRS', 'H2:CO2', np.float64(30.0))
Condition: ('BRRS', 'H2:CO2 + MeOH', np.float64(5.0))
Condition: ('BRRS', 'H2:CO2 + MeOH', np.float64(30.0))
Condition: ('BRRS', 'H2:CO2 + TMA', np.float64(5.0))
Condition: ('BRRS', 'H2:CO2 + TMA', np.float64(30.0))
Condition: ('OSU', 'H2:CO2', np.float64(5.0))
Condition: ('OSU', 'H2:CO2', np.float64(30.0))
Condition: ('OSU', 'H2:CO2 + MeOH', np.float64(5.0))
Condition: ('OSU', 'H2:CO2 + MeOH', np.float64(30.0))
Condition: ('OSU', 'H2:CO2 + TMA', np.float64(5.0))
Condition: ('OSU', 'H2:CO2 + TMA', np.float64(30.0))


In [15]:
len(strain_list)

751

In [16]:
len(condition_trajectories)

12